# Cycle Types in Young Double Cosets — companion notebook (draft v0.1)
Dean Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Every numbered claim marked [VERIFIED] in the paper is re-run here for $n\le7$. Set `RUN_LARGE = True` to extend the checks to $n=8$ (about a minute per cell) and $n=9$ for the table-only computations (several minutes). The scripts `audit.py`, `spectrum.py`, `countM.py`, `best.py` and `sweep8.py` must sit in the same folder as this notebook.

In [1]:
RUN_LARGE = False
import itertools, math, time
from collections import defaultdict, Counter
from math import factorial, prod, comb
from fractions import Fraction
from audit import int_partitions, ctype, UF, run as orbit_audit
from spectrum import tables, Lam, rel_counts
from countM import TS, Mvals, ec
from best import best_ncycles
from sweep8 import burnside

## Lemma 2.3 and Remark 8.2 — direct orbit counts, key invariance, inversion splitting, double-coset/table identity

In [2]:
for n in range(4, 8 if not RUN_LARGE else 8):
    orbit_audit(n)

n=4: cells 120  |X/S_n| 35  |X/(S_n x C2)| 35  key 35 (ambiguous 0, conj-orbits splitting key 0)  sym-key 35  values 6  conj+inv orbits split by key 0  cells with T not ~ T^t 0  double-coset<->T mismatches 0
n=5: cells 840  |X/S_n| 102  |X/(S_n x C2)| 101  key 97 (ambiguous 0, conj-orbits splitting key 0)  sym-key 97  values 13  conj+inv orbits split by key 0  cells with T not ~ T^t 0  double-coset<->T mismatches 0


n=6: cells 7920  |X/S_n| 395  |X/(S_n x C2)| 372  key 359 (ambiguous 0, conj-orbits splitting key 0)  sym-key 344  values 30  conj+inv orbits split by key 18  cells with T not ~ T^t 416  double-coset<->T mismatches 0


n=7: cells 75600  |X/S_n| 1480  |X/(S_n x C2)| 1297  key 1181 (ambiguous 0, conj-orbits splitting key 0)  sym-key 1086  values 60  conj+inv orbits split by key 122  cells with T not ~ T^t 7824  double-coset<->T mismatches 0


## Theorem 3.2 — non-containment census (Table 1)

In [3]:
def nsetparts(mu):
    c=Counter(mu); return factorial(sum(mu))//(prod(factorial(k) for k in mu)*prod(factorial(v) for v in c.values()))
for n in (5,6,7):
    perms=list(itertools.permutations(range(n))); lam=[ctype(p) for p in perms]; clsz=Counter(lam)
    cells=strict=ties=bad=0; gain=0.0
    for mu in int_partitions(n):
        if all(k==1 for k in mu): continue
        mult=nsetparts(mu); m=len(mu); b=[]
        for i,k in enumerate(mu): b+=[i]*k
        grp=defaultdict(Counter); Tof=[]
        for si,p in enumerate(perms):
            T=[[0]*m for _ in range(m)]
            for x in range(n): T[b[x]][b[p[x]]]+=1
            T=tuple(map(tuple,T)); Tof.append(T); grp[T][lam[si]]+=1
        Kf=prod(factorial(k) for k in mu)
        for si in range(len(perms)):
            T=Tof[si]; M=grp[T][lam[si]]; DC=Kf*Kf//prod(factorial(v) for r in T for v in r); Cl=clsz[lam[si]]
            cells+=mult; best=min(DC,Cl)
            if M<best: strict+=mult
            else:
                ties+=mult; bad+= mult if DC<=Cl else 0
            gain+=mult*(math.log2(best)-math.log2(M))
    print(f"n={n}: configurations {cells:,}  strict {strict:,}  ties {ties:,}  ties with |KsK|<=|C| {bad}  mean gain {gain/cells:.3f} bits")

n=5: configurations 6,120  strict 5,730  ties 390  ties with |KsK|<=|C| 0  mean gain 1.193 bits
n=6: configurations 145,440  strict 142,929  ties 2,511  ties with |KsK|<=|C| 0  mean gain 1.685 bits


n=7: configurations 4,415,040  strict 4,396,147  ties 18,893  ties with |KsK|<=|C| 0  mean gain 2.221 bits


## Theorem 4.2 / Example 4.5 — worked example

In [4]:
mu=(2,2); T=((1,1),(1,1))
print({l:int(v) for l,v in Mvals(T,mu).items()}, ' |KsK| =', prod(factorial(d) for d in mu)**2//prod(factorial(v) for r in T for v in r))

{(2, 1, 1): 4, (3, 1): 8, (4,): 4}  |KsK| = 16


## Theorems 4.2, 5.1, 6.1 — enumeration versus transition-system counting on every table

In [5]:
for n in range(5, 9 if RUN_LARGE else 8):
    t0=time.time(); perms=list(itertools.permutations(range(n))); lam=[ctype(p) for p in perms]
    tot=bad_rec=bad_best=bad_supp=0
    for mu in int_partitions(n):
        m=len(mu); b=[]
        for i,k in enumerate(mu): b+=[i]*k
        grp=defaultdict(Counter)
        for si,p in enumerate(perms):
            T=[[0]*m for _ in range(m)]
            for x in range(n): T[b[x]][b[p[x]]]+=1
            grp[tuple(map(tuple,T))][lam[si]]+=1
        for T,Cn in grp.items():
            tot+=1
            if {l:Fraction(v) for l,v in Cn.items()}!=Mvals(T,mu): bad_rec+=1
            if best_ncycles([list(r) for r in T],mu)!=Cn.get((n,),0): bad_best+=1
            if set(Cn)!=set(Lam(T)): bad_supp+=1
    print(f"n={n}: tables {tot}  recursion mismatches {bad_rec}  BEST mismatches {bad_best}  support mismatches {bad_supp}  [{time.time()-t0:.0f}s]")

n=5: tables 177  recursion mismatches 0  BEST mismatches 0  support mismatches 0  [0s]


n=6: tables 1054  recursion mismatches 0  BEST mismatches 0  support mismatches 0  [0s]


n=7: tables 7237  recursion mismatches 0  BEST mismatches 0  support mismatches 0  [4s]


## Theorem 5.3 — hook classes $(n-k,1^k)$

In [6]:
def hook_TS(T,k):
    m=len(T); tot=0
    for a in itertools.product(*[range(0,T[v][v]+1) for v in range(m)]):
        if sum(a)!=k: continue
        R=[list(r) for r in T]
        for v in range(m): R[v][v]-=a[v]
        tot+=prod(comb(T[v][v],a[v]) for v in range(m))*ec(R,m)
    return tot
tot=bad=0
for n in range(2,8):
    for mu in int_partitions(n):
        for T in tables(mu):
            MV=Mvals(T,mu); sc=Fraction(prod(factorial(d) for d in mu),prod(factorial(v) for r in T for v in r))
            for k in range(0,n-1):
                tot+=1; bad+= Fraction(hook_TS(T,k))*sc != MV.get(tuple([n-k]+[1]*k),0)
print(f"hook formula: {tot-bad}/{tot} exact")

hook formula: 49532/49532 exact


## Table 2 — Burnside orbit counts and cells computed from tables alone

In [7]:
for n in range(1, 10 if RUN_LARGE else 8):
    t0=time.time(); b1,b2=burnside(n) if n<=8 or RUN_LARGE else (None,None)
    ntab,c1,c2=rel_counts(n)
    print(f"n={n}: tables {ntab}  |X/S_n| {b1}  |X/(S_n x C2)| {b2}  cells {c1}  cells+inv {c2}  [{time.time()-t0:.0f}s]")

n=1: tables 1  |X/S_n| 1  |X/(S_n x C2)| 1  cells 1  cells+inv 1  [0s]
n=2: tables 3  |X/S_n| 4  |X/(S_n x C2)| 4  cells 4  cells+inv 4  [0s]
n=3: tables 9  |X/S_n| 10  |X/(S_n x C2)| 10  cells 10  cells+inv 10  [0s]
n=4: tables 37  |X/S_n| 35  |X/(S_n x C2)| 35  cells 35  cells+inv 35  [0s]
n=5: tables 177  |X/S_n| 102  |X/(S_n x C2)| 101  cells 97  cells+inv 97  [0s]
n=6: tables 1054  |X/S_n| 395  |X/(S_n x C2)| 372  cells 359  cells+inv 344  [0s]


n=7: tables 7237  |X/S_n| 1480  |X/(S_n x C2)| 1297  cells 1181  cells+inv 1086  [1s]


## Measurement 9.1 — distinct values of $F$ from the recursion

In [8]:
def values(n):
    vals=set()
    for mu in int_partitions(n):
        m=len(mu); tabs=tables(mu); ix={t:i for i,t in enumerate(tabs)}; u=UF(len(tabs))
        for t,i in ix.items():
            for s in [a for a in range(m-1) if mu[a]==mu[a+1]]:
                p=list(range(m)); p[s],p[s+1]=s+1,s
                u.u(i, ix[tuple(tuple(t[p[a]][p[c]] for c in range(m)) for a in range(m))])
            u.u(i, ix[tuple(tuple(t[c][a] for c in range(m)) for a in range(m))])
        reps={}
        for t,i in ix.items(): reps.setdefault(u.f(i),t)
        for t in reps.values(): vals|={int(v) for v in Mvals(t,mu).values()}
    return sorted(vals)
for n in range(1, 10 if RUN_LARGE else 8):
    v=values(n); print(f"n={n}: {len(v)} values" + (f": {v}" if n==7 else ""))

n=1: 1 values
n=2: 1 values
n=3: 3 values
n=4: 6 values
n=5: 13 values
n=6: 30 values


n=7: 60 values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 18, 20, 21, 24, 25, 28, 30, 32, 36, 40, 45, 48, 50, 54, 60, 64, 70, 72, 84, 90, 96, 100, 105, 108, 120, 140, 144, 160, 168, 210, 216, 240, 260, 280, 288, 300, 324, 360, 420, 432, 480, 504, 540, 630, 720, 840]
